# 텍스트 추출
## 라이브러리 설치

In [1]:
# !pip install pdfplumber

## 이메일 계정 접속 및 메일 조회

In [ ]:
import imaplib
import email
import os
import pdfplumber
from email.header import decode_header
from datetime import datetime, timedelta

EMAIL = "아이디"
PASSWORD = "앱_비밀번호"  # 앱 비밀번호 16자리

mail = imaplib.IMAP4_SSL("imap.gmail.com")
mail.login(EMAIL, PASSWORD)
mail.select("INBOX")

# ── 전체 메일 ──────────────────────────────
_, msg_nums = mail.search(None, "ALL")

# ── 최근 1일 ──────────────────────────────
# since_date = (datetime.now() - timedelta(days=1)).strftime("%d-%b-%Y")
# _, msg_nums = mail.search(None, f'SINCE {since_date}')

# ── 최근 7일 ──────────────────────────────
# since_date = (datetime.now() - timedelta(days=7)).strftime("%d-%b-%Y")
# _, msg_nums = mail.search(None, f'SINCE {since_date}')
num_list = msg_nums[0].split()

DOWNLOAD_DIR = "downloads"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

print(f"메일 수: {len(num_list)}건")

메일 수: 13건


## 디코딩 함수

In [3]:
def decode_str(s):
    if s is None:
        return ""
    result = ""
    for part, enc in decode_header(s):
        if isinstance(part, bytes):
            result += part.decode(enc or "utf-8", errors="ignore")
        else:
            result += str(part)
    return result

## PDF 텍스트 추출 함수

In [4]:
def extract_pdf_text(filepath):
    text = ""
    try:
        with pdfplumber.open(filepath) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
    except Exception as e:
        text = f"[추출 실패: {e}]"
    return text.strip()

## 제목 / 본문 / PDF 텍스트 추출

In [5]:
CATEGORIES = {
    "업무협조": ["요청", "협조", "부탁", "전달", "회신", "처리", "검토", "승인"],
    "보고서":   ["보고", "결과", "분석", "현황", "매출", "데이터", "통계", "성과"],
    "회의록":   ["회의", "안건", "참석자", "논의", "결정", "회의록"],
    "공지":     ["공지", "안내", "변경", "배포", "적용"],
}

TAG_MAP = {
    "[업무협조]": "업무협조",
    "[보고]":     "보고서",
    "[회의록]":   "회의록",
    "[공지]":     "공지",
}

THRESHOLD = 0.5  # 1위 카테고리가 50% 미만이면 기타

def classify_category(subject, body):
    # 1. 제목 태그 우선 분류
    for tag, cat in TAG_MAP.items():
        if tag in subject:
            return cat

    # 2. 키워드 점수 계산
    text = subject + " " + body
    scores = {cat: sum(text.count(kw) for kw in kws)
              for cat, kws in CATEGORIES.items()}
    total = sum(scores.values())

    # 3. 키워드 하나도 없으면 기타
    if total == 0:
        return "기타"

    # 4. 1위 카테고리 신뢰도가 낮으면 기타
    best = max(scores, key=scores.get)
    confidence = scores[best] / total

    if confidence < THRESHOLD:
        return "기타"

    return best

# 카테고리 폴더 미리 생성
for cat in ["업무협조", "보고서", "회의록", "공지", "기타"]:
    os.makedirs(os.path.join(DOWNLOAD_DIR, cat), exist_ok=True)

results = []

for num in num_list[::-1]:
    _, data = mail.fetch(num, "(RFC822)")
    msg = email.message_from_bytes(data[0][1])

    # 1. 제목
    subject = decode_str(msg["Subject"])

    # 발신자 
    sender = decode_str(msg["From"])

    # 2. 본문
    body = ""
    for part in msg.walk():
        if part.get_content_type() == "text/plain":
            if "attachment" not in str(part.get("Content-Disposition", "")):
                try:
                    body = part.get_payload(decode=True).decode("utf-8", errors="ignore")
                    break
                except:
                    pass

    # 3. 1차 분류
    category = classify_category(subject, body)

    # 4. PDF를 카테고리 폴더에 바로 저장
    pdf_text = ""
    pdfs = []
    for part in msg.walk():
        disposition = str(part.get("Content-Disposition", ""))
        filename = part.get_filename()
        if "attachment" in disposition and filename:
            filename = decode_str(filename)
            if filename.lower().endswith(".pdf"):
                filepath = os.path.join(DOWNLOAD_DIR, category, filename)
                with open(filepath, "wb") as f:
                    f.write(part.get_payload(decode=True))
                pdf_text += extract_pdf_text(filepath)
                pdfs.append(filepath)

    results.append({
        "subject":      subject,
        "sender":       sender,
        "body":         body.strip(),
        "pdf_text":     pdf_text.strip(),
        "pdf_paths":    pdfs,
        "category_1차": category,
    })

print(f"✅ {len(results)}건 추출 완료")

# 카테고리별 현황 출력
from collections import Counter
cat_counts = Counter(r["category_1차"] for r in results)
for cat, count in cat_counts.items():
    print(f"  {cat:10} : {count}건")

✅ 13건 추출 완료
  보고서        : 4건
  공지         : 1건
  회의록        : 1건
  업무협조       : 5건
  기타         : 2건


## 결과 확인

In [6]:
import pandas as pd

df = pd.DataFrame(results)
df["body_len"] = df["body"].apply(len)
df["pdf_len"] = df["pdf_text"].apply(len)

print(df[["subject", "body_len", "pdf_len"]])

                                           subject  body_len  pdf_len
0                [보고서] 4월 서비스 데이터 분석 종합 보고서 초안입니다.        33     1611
1                                     일정 변경 공지입니다.        56      128
2                              참석자분들께 회의록 배포해드립니다.        12      163
3                                    회의 결과 보고드립니다.        25      230
4                            안내받은 바에 따라 협조 요청드립니다.        21      353
5                                            보안 알림       981        0
6        중요한 공지사항에 관한 보고 결과입니다. 확인 후 회신 협조 요청 드립니다         8      874
7                                  프라치노 공책 거래 영수증t         0        0
8                                        프라치노 공책주문         0        0
9                             Re: [업무협조] 데이터 검증 요청       157        0
10                                [업무협조] 데이터 검증 요청        54      116
11                                 [업무협조] 기능 검토 요청        60      143
12  발표자료 - 배경 따릉이 이용 데이터는 공개되어 있지만, CSV 형태의 원시 ...       918        0


## 미리보기

In [7]:
sample = results[0]
print("=== 제목 ===")
print(sample["subject"])
print("\n=== 본문 ===")
print(sample["body"][:300])
print("\n=== PDF 텍스트 ===")
print(sample["pdf_text"][:300])
print("\n=== PDF 경로 ===")
print(sample["pdf_paths"])

=== 제목 ===
[보고서] 4월 서비스 데이터 분석 종합 보고서 초안입니다.

=== 본문 ===
검토 후 5/13(수)까지 회신 부탁드립니다.
감사합니다.

=== PDF 텍스트 ===
[2026년 4월 서비스 데이터 분석 종합 보고서]
작성부서: 데이터분석팀
작성일자: 2026-04-30
1. 보고 목적
본 보고서는 2026년 4월 한 달간 서비스 이용 데이터를 종합적으로 분석하여
주요 성과 지표(KPI), 사용자 행동 패턴, 이상 징후 및 개선 필요 사항을 도출하고,
향후 서비스 운영 및 전략 수립에 참고 자료로 활용하기 위함이다.
2. 분석 개요
2.1 분석 기간
- 2026년 4월 1일 ~ 2026년 4월 30일
2.2 분석 대상
- 웹 및 모바일 서비스 전체 사용자
2.3 분석 도구
- 내부 로그 시스템


=== PDF 경로 ===
['downloads\\보고서\\26년 4월 서비스 데이터 분석 종합 보고서.pdf']


 ## results.json 저장

In [8]:
import json

with open("results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f" results.json 저장 완료 ({len(results)}건)")

 results.json 저장 완료 (13건)


## 접속 종료

In [ ]:
mail.logout()
print("접속 종료")

접속 종료


: 